In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import sys

In [3]:
spark = SparkSession.builder \
        .appName("FAOSTAT Data Ingestion") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.lakehouse.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
        .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://localhost:5432/metastore") \
        .config("spark.sql.catalog.lakehouse.jdbc.user", "admin") \
        .config("spark.sql.catalog.lakehouse.jdbc.password", "password") \
        .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.hadoop:hadoop-aws:3.3.4,org.postgresql:postgresql:42.6.0,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
        .getOrCreate()

In [8]:
spark.sql("show tables from lakehouse.bronze").show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|   bronze|faostat_qcl_raw|      false|
|   bronze|faostat_tcl_raw|      false|
+---------+---------------+-----------+



## TAB 1 -- PRODUCTION CROPS LIVESTOCK

In [9]:
spark.sql("select * from lakehouse.bronze.faostat_qcl_raw limit 5").show()

+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|Area Code|Area Code (M49)|       Area|Item Code|Item Code (CPC)|             Item|Element Code|       Element|Year Code|Year|Unit|Value|Flag|Note|      ingestion_time|
+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5312|Area harvested|     1961|1961|  ha|  0.0|   A|NULL|2026-03-17 04:12:...|
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5312|Area harvested|     1962|1962|  ha|  0.0|   A|NULL|2026-03-17 04:12:...|
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5312|Area harvested|     1963|1963|  ha|  0.0|   A|NULL|2026-03-

In [14]:
spark.sql("""
SELECT DISTINCT YEAR 
FROM lakehouse.bronze.faostat_qcl_raw
ORDER BY YEAR ASC
LIMIT 10
""").show(truncate=False, vertical=True)

-RECORD 0----
 YEAR | 1961 
-RECORD 1----
 YEAR | 1962 
-RECORD 2----
 YEAR | 1963 
-RECORD 3----
 YEAR | 1964 
-RECORD 4----
 YEAR | 1965 
-RECORD 5----
 YEAR | 1966 
-RECORD 6----
 YEAR | 1967 
-RECORD 7----
 YEAR | 1968 
-RECORD 8----
 YEAR | 1969 
-RECORD 9----
 YEAR | 1970 



In [ ]:
spark.sql("""
SELECT COUNT (DISTINCT Area)
FROM lakehouse.bronze.faostat_qcl_raw
LIMIT 10
""").show(truncate=False, vertical=True)

-RECORD 0-------------------
 count(DISTINCT Area) | 244 



In [31]:
spark.sql("""
SELECT DISTINCT 
    Item
FROM lakehouse.bronze.faostat_qcl_raw
WHERE Element = 'Production'
LIMIT 10
""").show(truncate=False)

+-------------------------------------------------+
|Item                                             |
+-------------------------------------------------+
|Butter of cow milk                               |
|Olive oil                                        |
|Whey, dry                                        |
|Rapeseed or canola oil, crude                    |
|Canary seed                                      |
|Meat of other domestic camelids, fresh or chilled|
|Sheep and Goat Meat                              |
|Pineapples                                       |
|Potatoes                                         |
|Roots and Tubers, Total                          |
+-------------------------------------------------+



In [27]:
spark.sql("""
SELECT COUNT (DISTINCT Item)
FROM lakehouse.bronze.faostat_qcl_raw
WHERE Element = 'Production'
LIMIT 10
""").show(truncate=False, vertical=True)

-RECORD 0-------------------
 count(DISTINCT Item) | 280 



In [ ]:
spark.sql("""
WITH unique_items AS (
    SELECT DISTINCT Item 
    FROM lakehouse.bronze.faostat_qcl_raw 
    WHERE Element = 'Production'
),
total_count AS (
    SELECT COUNT(*) as total 
    FROM unique_items
)
SELECT 
    u.Item, 
    (SELECT total FROM total_count) as total_unique_items
FROM unique_items u
LIMIT 10
""").show(truncate=False)

+-------------------------------------------------+------------------+
|Item                                             |total_unique_items|
+-------------------------------------------------+------------------+
|Butter of cow milk                               |280               |
|Olive oil                                        |280               |
|Whey, dry                                        |280               |
|Rapeseed or canola oil, crude                    |280               |
|Canary seed                                      |280               |
|Meat of other domestic camelids, fresh or chilled|280               |
|Sheep and Goat Meat                              |280               |
|Pineapples                                       |280               |
|Potatoes                                         |280               |
|Roots and Tubers, Total                          |280               |
+-------------------------------------------------+------------------+



In [ ]:
spark.sql("""
    SELECT DISTINCT Unit 
    FROM lakehouse.bronze.faostat_qcl_raw 
    WHERE Element = 'Production'
""").show(truncate=False, vertical=True)

+-------+
|Unit   |
+-------+
|1000 No|
|t      |
+-------+



In [35]:
spark.sql("""
    SELECT Unit, COUNT(*)  
    FROM lakehouse.bronze.faostat_qcl_raw 
    WHERE Element = 'Production'
    GROUP BY Unit
""").show(truncate=False, vertical=True)

-RECORD 0-----------
 Unit     | 1000 No 
 count(1) | 18208   
-RECORD 1-----------
 Unit     | t       
 count(1) | 1625403 



Using DATA FRAME

In [36]:
raw_df = spark.read.table("lakehouse.bronze.faostat_qcl_raw")

In [46]:
production_df = raw_df.filter(col("Element")== "Production") \
    .select(
        col("Area").alias("country"),
        col("item").alias("crop"),
        col("Year").cast("int").alias("year"),
        col("Value").cast("double").alias("production_tonnes"),
        col("Unit").alias("unit")
    ) 

In [39]:
production_df.show(5, truncate=False)

+-----------+-----------------+----+-----------------+----+
|country    |crop             |year|production_tonnes|unit|
+-----------+-----------------+----+-----------------+----+
|Afghanistan|Almonds, in shell|1961|0.0              |t   |
|Afghanistan|Almonds, in shell|1962|0.0              |t   |
|Afghanistan|Almonds, in shell|1963|0.0              |t   |
|Afghanistan|Almonds, in shell|1964|0.0              |t   |
|Afghanistan|Almonds, in shell|1965|0.0              |t   |
+-----------+-----------------+----+-----------------+----+
only showing top 5 rows



In [43]:
production_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- crop: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- production_tonnes: double (nullable = true)
 |-- unit: string (nullable = true)



In [47]:
from pyspark.sql.functions import sum 
yearly_global = production_df.groupBy("year", "crop").agg(sum("production_tonnes").alias("total_production"))

In [49]:
yearly_global.show(10, truncate=False)

+----+---------------------------------------------+--------------------+
|year|crop                                         |total_production    |
+----+---------------------------------------------+--------------------+
|1982|Almonds, in shell                            |4863026.0           |
|1996|Almonds, in shell                            |5852574.770000001   |
|1967|Butter and ghee of sheep milk                |250147.54000000004  |
|1979|Butter of cow milk                           |2.6634119100000005E7|
|2021|Cattle fat, unrendered                       |1.390722789E7       |
|1961|Cheese from milk of sheep, fresh or processed|1916286.6500000001  |
|2021|Cheese from milk of sheep, fresh or processed|2714991.53          |
|2005|Grapes                                       |3.0755777946E8      |
|1997|Hen eggs in shell, fresh                     |4.166646726489999E9 |
|2014|Meat of camels, fresh or chilled             |3046765.5700000003  |
+----+--------------------------------

In [75]:
top_crops = production_df.groupBy("crop").agg(sum("production_tonnes").alias("total_production")) \
    .orderBy(col("total_production").desc()) \
    .limit(20)

In [76]:
top_crops.show(truncate=False)

+------------------------------+---------------------+
|crop                          |total_production     |
+------------------------------+---------------------+
|Cereals, primary              |5.7185347004783E11   |
|Sugar Crops Primary           |3.9754631556151996E11|
|Sugar cane                    |3.231591264832504E11 |
|Hen eggs in shell, fresh      |2.516557523842102E11 |
|Roots and Tubers, Total       |1.998724122336801E11 |
|Vegetables Primary            |1.7472151587101007E11|
|Maize (corn)                  |1.717477342585899E11 |
|Milk, Total                   |1.666192509922201E11 |
|Rice                          |1.559206248910801E11 |
|Wheat                         |1.5315627567287012E11|
|Fruit Primary                 |1.5261546497652017E11|
|Raw milk of cattle            |1.4223613403094006E11|
|Potatoes                      |9.011076274888995E10 |
|Sugar beet                    |7.409734343813E10    |
|Meat, Total                   |5.936201613812998E10 |
|Cassava, 

Using Spark Sql

In [68]:
production_df = spark.sql("""
    SELECT Area AS country, item AS crop, Year AS year, Value AS production_tonnes, Unit AS unit
    FROM lakehouse.bronze.faostat_qcl_raw
    WHERE Element = 'Production'
""")

In [72]:
production_df.show(5, truncate=False)

+-----------+-----------------+----+-----------------+----+
|country    |crop             |year|production_tonnes|unit|
+-----------+-----------------+----+-----------------+----+
|Afghanistan|Almonds, in shell|1961|0.0              |t   |
|Afghanistan|Almonds, in shell|1962|0.0              |t   |
|Afghanistan|Almonds, in shell|1963|0.0              |t   |
|Afghanistan|Almonds, in shell|1964|0.0              |t   |
|Afghanistan|Almonds, in shell|1965|0.0              |t   |
+-----------+-----------------+----+-----------------+----+
only showing top 5 rows



In [73]:
production_df.createOrReplaceTempView("production_df")

yearly_global = spark.sql("""
    SELECT year, crop, SUM(production_tonnes) AS total_production
    FROM production_df
    GROUP BY year, crop
""")


In [74]:
yearly_global.show(10, truncate=False)

+----+---------------------------------------------+--------------------+
|year|crop                                         |total_production    |
+----+---------------------------------------------+--------------------+
|1982|Almonds, in shell                            |4863026.0           |
|1996|Almonds, in shell                            |5852574.770000001   |
|1967|Butter and ghee of sheep milk                |250147.54000000004  |
|1979|Butter of cow milk                           |2.6634119100000005E7|
|2021|Cattle fat, unrendered                       |1.390722789E7       |
|1961|Cheese from milk of sheep, fresh or processed|1916286.6500000001  |
|2021|Cheese from milk of sheep, fresh or processed|2714991.53          |
|2005|Grapes                                       |3.0755777946E8      |
|1997|Hen eggs in shell, fresh                     |4.166646726489999E9 |
|2014|Meat of camels, fresh or chilled             |3046765.5700000003  |
+----+--------------------------------

In [77]:
top_crops = spark.sql("""
    SELECT crop, SUM(production_tonnes) AS total_production
    FROM production_df
    GROUP BY crop
    ORDER BY total_production DESC
    LIMIT 20
""")

In [78]:
top_crops.show(truncate=False)

+------------------------------+---------------------+
|crop                          |total_production     |
+------------------------------+---------------------+
|Cereals, primary              |5.7185347004783E11   |
|Sugar Crops Primary           |3.9754631556151996E11|
|Sugar cane                    |3.231591264832504E11 |
|Hen eggs in shell, fresh      |2.516557523842102E11 |
|Roots and Tubers, Total       |1.998724122336801E11 |
|Vegetables Primary            |1.7472151587101007E11|
|Maize (corn)                  |1.717477342585899E11 |
|Milk, Total                   |1.666192509922201E11 |
|Rice                          |1.559206248910801E11 |
|Wheat                         |1.5315627567287012E11|
|Fruit Primary                 |1.5261546497652017E11|
|Raw milk of cattle            |1.4223613403094006E11|
|Potatoes                      |9.011076274888995E10 |
|Sugar beet                    |7.409734343813E10    |
|Meat, Total                   |5.936201613812998E10 |
|Cassava, 

## TAB 2 -- TRADE CROPS LIVESTOCK

In [ ]:
spark.sql("select * from lakehouse.bronze.faostat_tcl_raw limit 5").show()

+---------+---------------+-----------+---------+---------------+-----------------+------------+---------------+---------+----+----+------+----+--------------------+--------------------+
|Area Code|Area Code (M49)|       Area|Item Code|Item Code (CPC)|             Item|Element Code|        Element|Year Code|Year|Unit| Value|Flag|                Note|      ingestion_time|
+---------+---------------+-----------+---------+---------------+-----------------+------------+---------------+---------+----+----+------+----+--------------------+--------------------+
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5610|Import quantity|     2014|2014|   t| 34.46|   X|Estimated data us...|2026-03-17 04:14:...|
|        3|           '008|    Albania|      221|         '01371|Almonds, in shell|        5610|Import quantity|     1996|1996|   t|  87.0|   A|                NULL|2026-03-17 04:14:...|
|        2|           '004|Afghanistan|      221|         '01371|

In [5]:
spark.sql("""
    SELECT *
    FROM lakehouse.bronze.faostat_tcl_raw
    LIMIT 1
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------
 Area Code       | 2                                              
 Area Code (M49) | '004                                           
 Area            | Afghanistan                                    
 Item Code       | 221                                            
 Item Code (CPC) | '01371                                         
 Item            | Almonds, in shell                              
 Element Code    | 5610                                           
 Element         | Import quantity                                
 Year Code       | 2014                                           
 Year            | 2014                                           
 Unit            | t                                              
 Value           | 34.46                                          
 Flag            | X                                              
 Note            | Estimated data using trading partners datab

In [ ]:
spark.sql("""
    SELECT MIN(Year) AS min_year, MAX(Year) AS max_year
    FROM lakehouse.bronze.faostat_tcl_raw
""").show(truncate=False, vertical=True)

-RECORD 0--------
 min_year | 1961 
 max_year | 2024 



In [9]:
spark.sql("""
    SELECT DISTINCT Unit
    FROM lakehouse.bronze.faostat_tcl_raw
""").show(truncate=False, vertical=True)

-RECORD 0--------
 Unit | No       
-RECORD 1--------
 Unit | 1000 USD 
-RECORD 2--------
 Unit | 1000 An  
-RECORD 3--------
 Unit | t        
-RECORD 4--------
 Unit | An       



In [10]:
spark.sql("""
    SELECT DISTINCT Element
    FROM lakehouse.bronze.faostat_tcl_raw
""").show(truncate=False, vertical=True)

-RECORD 0------------------
 Element | Import quantity 
-RECORD 1------------------
 Element | Export quantity 
-RECORD 2------------------
 Element | Export value    
-RECORD 3------------------
 Element | Import value    



In [12]:
spark.sql("""
    SELECT *
    FROM lakehouse.bronze.faostat_tcl_raw
    WHERE Element = 'Import quantity'
    LIMIT 1
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------
 Area Code       | 2                                              
 Area Code (M49) | '004                                           
 Area            | Afghanistan                                    
 Item Code       | 221                                            
 Item Code (CPC) | '01371                                         
 Item            | Almonds, in shell                              
 Element Code    | 5610                                           
 Element         | Import quantity                                
 Year Code       | 2014                                           
 Year            | 2014                                           
 Unit            | t                                              
 Value           | 34.46                                          
 Flag            | X                                              
 Note            | Estimated data using trading partners datab

In [13]:
spark.sql("""
    SELECT *
    FROM lakehouse.bronze.faostat_tcl_raw
    WHERE Element = 'Import value'
    LIMIT 1
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------
 Area Code       | 2                                              
 Area Code (M49) | '004                                           
 Area            | Afghanistan                                    
 Item Code       | 221                                            
 Item Code (CPC) | '01371                                         
 Item            | Almonds, in shell                              
 Element Code    | 5622                                           
 Element         | Import value                                   
 Year Code       | 2014                                           
 Year            | 2014                                           
 Unit            | 1000 USD                                       
 Value           | 156.0                                          
 Flag            | X                                              
 Note            | Estimated data using trading partners datab

Using Data Frame

In [14]:
raw_df = spark.read.table("lakehouse.bronze.faostat_tcl_raw")

In [15]:
trade_df = raw_df.select(
    col("Area").alias("country"),
    col("item").alias("product"),
    col("Year").cast("int").alias("year"),
    col("Value").alias("double").alias("trade_value"),
    col("Element").alias("trade_type"),
).filter(col("trade_value").isNotNull())


In [17]:
trade_df.show(truncate=False)

+-------------------+-----------------------+----+-----------+---------------+
|country            |product                |year|trade_value|trade_type     |
+-------------------+-----------------------+----+-----------+---------------+
|Afghanistan        |Almonds, in shell      |2014|34.46      |Import quantity|
|Albania            |Almonds, in shell      |1996|87.0       |Import quantity|
|Afghanistan        |Almonds, in shell      |2015|75.5       |Import quantity|
|Albania            |Almonds, in shell      |1997|0.0        |Import quantity|
|Afghanistan        |Almonds, in shell      |2016|309.16     |Import quantity|
|Algeria            |Almonds, in shell      |1961|0.0        |Import quantity|
|Afghanistan        |Almonds, in shell      |2018|822.2      |Import quantity|
|Algeria            |Almonds, in shell      |1962|0.0        |Import quantity|
|Albania            |Almonds, in shell      |1998|0.0        |Import quantity|
|Angola             |Abaca, manila hemp, raw|2021|0.

Using Spark Sql

In [21]:
trade_df = spark.sql("""
    SELECT Area as country, item as product, Year as year, Value as trade_value, Element as trade_type
    FROM lakehouse.bronze.faostat_tcl_raw
    WHERE Value IS NOT NULL
""")

In [22]:
trade_df.show(5, truncate=False)

+-----------+-----------------+----+-----------+---------------+
|country    |product          |year|trade_value|trade_type     |
+-----------+-----------------+----+-----------+---------------+
|Afghanistan|Almonds, in shell|2014|34.46      |Import quantity|
|Albania    |Almonds, in shell|1996|87.0       |Import quantity|
|Afghanistan|Almonds, in shell|2015|75.5       |Import quantity|
|Albania    |Almonds, in shell|1997|0.0        |Import quantity|
|Afghanistan|Almonds, in shell|2016|309.16     |Import quantity|
+-----------+-----------------+----+-----------+---------------+
only showing top 5 rows

